# 02 · Silver: posted sales and master data
Typed, quarantined, deduplicated and MERGEd with a watermark. Ends with a reconciliation of daily net sales against Business Central's control totals.

In [ ]:
%run 00_common

In [ ]:
TS = "CAST(SystemModifiedAt AS TIMESTAMP) AS SystemModifiedAt"

load_silver("bronze_sales_invoice_header", "silver_sales_invoice_header", ["Company", "No"], f"""
    Company, No, Order_No, Sell_to_Customer_No,
    CAST(Order_Date AS DATE) AS Order_Date, CAST(Posting_Date AS DATE) AS Posting_Date,
    Location_Code, Salesperson_Code,
    Global_Dimension_1_Code AS Region_Code, Global_Dimension_2_Code AS Business_Unit, {TS}""")

line_cols = f"""
    Company, Document_No, CAST(Line_No AS INT) AS Line_No, No AS Item_No, Description, Location_Code,
    TRY_CAST(Quantity AS DECIMAL(18,4))             AS Quantity,
    TRY_CAST(Unit_Price AS DECIMAL(18,2))           AS Unit_Price,
    TRY_CAST(Line_Discount_Pct AS DECIMAL(9,4))     AS Line_Discount_Pct,
    TRY_CAST(Amount AS DECIMAL(18,2))               AS Amount,
    TRY_CAST(Amount_Including_VAT AS DECIMAL(18,2)) AS Amount_Incl_GST, {TS}"""
line_rules = [("Quantity IS NULL", "Quantity blank or not numeric"),
              ("Amount IS NULL",   "Amount blank or not numeric")]

load_silver("bronze_sales_invoice_line", "silver_sales_invoice_line",
            ["Company", "Document_No", "Line_No"], line_cols, line_rules)

load_silver("bronze_sales_cr_memo_header", "silver_sales_cr_memo_header", ["Company", "No"], f"""
    Company, No, Applies_to_Doc_No, Sell_to_Customer_No, CAST(Posting_Date AS DATE) AS Posting_Date,
    Return_Reason_Code, Location_Code,
    Global_Dimension_1_Code AS Region_Code, Global_Dimension_2_Code AS Business_Unit, {TS}""")

load_silver("bronze_sales_cr_memo_line", "silver_sales_cr_memo_line",
            ["Company", "Document_No", "Line_No"], line_cols, line_rules)

load_silver("bronze_item", "silver_item", ["Company", "No"], f"""
    Company, No, Description, Item_Category_Code, GTIN, Base_Unit_of_Measure,
    CAST(Unit_Price AS DECIMAL(18,2)) AS Unit_Price, CAST(Unit_Cost AS DECIMAL(18,2)) AS Unit_Cost,
    VAT_Prod_Posting_Group AS GST_Group, Vendor_No, {TS}""")

load_silver("bronze_location", "silver_location", ["Company", "Code"], f"""
    Company, Code, Name, Location_Type, City, State, {TS}""")

In [ ]:
spark.read.option("header", True).csv("Files/reference/bc_control_totals.csv").createOrReplaceTempView("bc_control")

spark.sql("""
WITH inv AS (
  SELECT h.Company, h.Posting_Date, SUM(l.Amount) AS amt
  FROM silver_sales_invoice_header h
  JOIN silver_sales_invoice_line l ON l.Company = h.Company AND l.Document_No = h.No
  GROUP BY 1, 2),
cm AS (
  SELECT h.Company, h.Posting_Date, SUM(l.Amount) AS amt
  FROM silver_sales_cr_memo_header h
  JOIN silver_sales_cr_memo_line l ON l.Company = h.Company AND l.Document_No = h.No
  GROUP BY 1, 2),
mine AS (
  SELECT COALESCE(i.Company, c.Company) AS Company,
         COALESCE(i.Posting_Date, c.Posting_Date) AS Posting_Date,
         COALESCE(i.amt, 0) - COALESCE(c.amt, 0) AS my_net_sales
  FROM inv i FULL OUTER JOIN cm c ON i.Company = c.Company AND i.Posting_Date = c.Posting_Date)
SELECT COUNT(*) AS days_checked,
       SUM(CASE WHEN ABS(m.my_net_sales - CAST(b.Net_Sales_Excl_GST AS DECIMAL(18,2))) > 0.01
                THEN 1 ELSE 0 END) AS days_not_matching
FROM bc_control b
LEFT JOIN mine m ON m.Company = b.Company AND m.Posting_Date = CAST(b.Posting_Date AS DATE)
""").show()